# 03 · Análisis del Segmento de Clientes
> **Pipeline:** Carga de resultados → Perfilamiento de clusters → Estadísticas descriptivas → Tabla resumen → Exportación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import datetime
import unicodedata
import re

## 1. Carga de datos etiquetados

In [ ]:
# Cargar CSV de características de clientes
ruta = "resultado_etiquetado.csv"
df = pd.read_csv(ruta, low_memory=False)

# --- Normalizar nombres de columnas al estándar esperado por el notebook ---
df = df.rename(columns={
    'Cluster':         'cluster',
    'Fechanacimiento': 'FechaNacimiento',
    'Estadocivil':     'EstadoCivil',
    'Tienepadres':     'TienePadres',
    'Tieneesposa':     'TieneEsposa',
    'Tienehijos':      'TieneHijos',
    'Valortotalplan':  'ValorTotalPlan',
    'Valormensual':    'ValorMensual',
    'Nroentidad':      'NroEntidad',
})

# --- Crear columnas numéricas (_V) requeridas por el análisis ---
# Binarias: Y → 1, N → 0
df['TienePadres_V'] = (df['TienePadres'] == 'Y').astype(int)
df['TieneEsposa_V'] = (df['TieneEsposa'] == 'Y').astype(int)
df['TieneHijos_V']  = (df['TieneHijos']  == 'Y').astype(int)

# Estado civil: codificación ordinal (CAS=0, DIV=1, OTR=2, SEP=3, SOL=4, UNI=5, VIU=6)
_ec_map = {'CAS': 0, 'DIV': 1, 'OTR': 2, 'SEP': 3, 'SOL': 4, 'UNI': 5, 'VIU': 6}
df['EstadoCivil_V'] = df['EstadoCivil'].map(_ec_map).fillna(2).astype(int)

# Región: codificación ordinal (AMAZONIA=0, ANDINA=1, CARIBE=2, ORINOQUIA=3, OTROS=4, PACIFICO=5)
_reg_map = {'AMAZONIA': 0, 'ANDINA': 1, 'CARIBE': 2, 'ORINOQUIA': 3, 'OTROS': 4, 'PACIFICO': 5}
df['Region_V'] = df['REGION'].map(_reg_map).fillna(4).astype(int)

print(f"Shape: {df.shape}")
print(f"Clusters: {sorted(df['cluster'].unique())}")
df.head(5)

In [ ]:
df.columns

In [ ]:
df['Estado'].value_counts()

## 2. Exploración inicial de clusters

In [ ]:
df['cluster'].value_counts()

In [ ]:
df["tamano_grupo"] = df.groupby("Entidad")["Contrato"].transform("count")

## 3. Preparación de variables para análisis

In [ ]:
df_modelo = df.copy()
# ValorTotal_scaled ya viene escalado desde NB01 (StandardScaler de Valortotalplan)
# No re-escalar: conservar la escala original del pipeline

In [ ]:
df_modelo["Año_nacimiento"] = pd.to_datetime(
    df_modelo["FechaNacimiento"],
    errors="coerce"
).dt.year

In [ ]:
# Variables de análisis: demografía + variables del modelo K-Prototypes (NB02)
# Modelo usó (13 vars): Total_afiliados, Cuotas, Cantidad_mascotas, Edad,
#   ValorTotal_scaled, Estrato, Canal, Producto, REGION, Sexo,
#   TienePadres_V, Estadocivil, tiene_nucleo_familiar
columnas_kmeans = [
    # Demográficas
    "Sexo",
    "Edad",
    "Año_nacimiento",
    "EstadoCivil_V",
    "TienePadres_V",
    "TieneEsposa_V",
    "TieneHijos_V",
    "Region_V",
    # Variables del modelo K-Prototypes (NB02)
    "Estrato",
    "Cantidad_mascotas",
    "Cuotas",
    "ValorTotalPlan",
    "ValorTotal_scaled",
    # Comerciales (también en modelo)
    "Producto",
    "Canal",
    # Derivadas
    "tamano_grupo",
    "cluster"
]

df_kmeans = df_modelo[columnas_kmeans]

## 4. Visualización de distribución por cluster

In [ ]:
#Graficar los valores en un BloxPot
df_filtrado = df_kmeans[df_kmeans['ValorTotal_scaled'] > 0]

plt.figure(figsize=(6,5))
plt.boxplot(df_kmeans['ValorTotal_scaled'], vert=True)
plt.title('Distribución Mensual (sin ceros)')
plt.ylabel('Valor mensual ($)')
plt.show()

In [ ]:
cluster_pct = (
    df_kmeans["cluster"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("pct_cluster")
)

In [ ]:
perfil_clusters = (
    df_kmeans
    .groupby('cluster')
    .mean(numeric_only=True)
    .round(2)
)

perfil_clusters

In [ ]:
#Plotting countplot of clusters
pal = ["#682F2F","#B9C0C9", "#9F8A78"]
pl = sns.countplot(x=df_kmeans["cluster"], hue=df_kmeans["cluster"],
                   palette=pal, legend=False)
pl.set_title("Distribution Of The Clusters")
plt.show()


## 5. Estadísticas descriptivas por cluster

In [ ]:
df_res = round(df['cluster'].value_counts(normalize = True),2).rename_axis('group').to_frame('group_size').sort_index()
df_res

In [ ]:
df_kmeans.groupby('cluster')[['Sexo']].describe()

In [ ]:
df_kmeans.groupby('cluster')[['Edad']].describe()

In [ ]:
df_kmeans.groupby('cluster')[['EstadoCivil_V']].describe()

In [ ]:
df_kmeans.groupby('cluster')[['TienePadres_V']].describe()

In [ ]:
df_kmeans.groupby('cluster')[['TieneEsposa_V']].describe()

In [ ]:
df_kmeans.groupby('cluster')[['TieneHijos_V']].describe()

In [ ]:
df_kmeans.groupby('cluster')[['Region_V']].describe()

In [ ]:
df_kmeans.groupby('cluster')[['ValorTotal_scaled']].describe()

In [ ]:
df_kmeans.groupby('cluster')[['ValorTotalPlan']].describe()

In [ ]:
# Variables clave del modelo K-Prototypes (NB02): Estrato, Cantidad_mascotas, Cuotas
df_kmeans.groupby('cluster')[['Estrato']].describe()

In [ ]:
df_kmeans.groupby('cluster')[['Cantidad_mascotas']].describe()

In [ ]:
df_kmeans.groupby('cluster')[['Cuotas']].describe()

### Distribución de `tiene_nucleo_familiar` por cluster
> Variable fusionada en NB02: reemplaza TieneEsposa_V + TieneHijos_V (PhiK=0.891)

In [ ]:
# tiene_nucleo_familiar: SI si tiene esposa O hijos, NO si ninguno
df['tiene_nucleo_familiar'] = (
    (df['TieneEsposa_V'] == 1) | (df['TieneHijos_V'] == 1)
).map({True: 'SI', False: 'NO'})

nf_dist = (
    df.groupby(['cluster', 'tiene_nucleo_familiar'])
    .size()
    .reset_index(name='n')
)
nf_dist['pct'] = (
    nf_dist.groupby('cluster')['n']
    .transform(lambda x: (x / x.sum() * 100).round(1))
)
print('Distribución tiene_nucleo_familiar por cluster:')
print(nf_dist.pivot(index='cluster', columns='tiene_nucleo_familiar', values='pct').fillna(0))


### Distribución de `Producto` y `Canal` por cluster

In [ ]:
for var in ['Producto', 'Canal']:
    print(f'\n=== {var} por Cluster ===')
    tbl = (
        df.groupby(['cluster', var])
        .size()
        .reset_index(name='n')
    )
    tbl['pct'] = (
        tbl.groupby('cluster')['n']
        .transform(lambda x: (x / x.sum() * 100).round(1))
    )
    pivot = tbl.pivot(index='cluster', columns=var, values='pct').fillna(0)
    print(pivot.to_string())


### Análisis de Antigüedad por cluster
> Proxy de fidelidad y riesgo de baja — mayor antigüedad indica cliente consolidado

In [ ]:
# Antigüedad en días desde FechaIngreso
fecha_col = 'Fechaingreso'
df[fecha_col] = pd.to_datetime(df[fecha_col], errors='coerce')
hoy = pd.Timestamp.today().normalize()
df['Antiguedad_dias'] = (hoy - df[fecha_col]).dt.days.clip(lower=0)

ant_stats = (
    df.groupby('cluster')['Antiguedad_dias']
    .describe(percentiles=[.25, .5, .75])
    [['mean', '25%', '50%', '75%']]
    .round(0).astype(int)
)
ant_stats['media_años'] = (ant_stats['mean'] / 365).round(1)
print('Antigüedad por cluster (días):')
print(ant_stats)

fig, ax = plt.subplots(figsize=(8, 4))
df.boxplot(column='Antiguedad_dias', by='cluster', ax=ax,
           patch_artist=True, medianprops=dict(color='red', linewidth=2))
ax.set_title('Antigüedad por cluster (días)')
ax.set_xlabel('Cluster'); ax.set_ylabel('Días')
plt.suptitle('')
plt.tight_layout(); plt.show()

### Distribución de Estado (ACT / RENVDO) por cluster
> Proporción de renovaciones por segmento — indica madurez y fidelidad del contrato

In [ ]:
if 'Estado' in df.columns:
    estado_dist = (
        df.groupby(['cluster', 'Estado'])
        .size()
        .reset_index(name='n')
    )
    estado_dist['pct'] = (
        estado_dist.groupby('cluster')['n']
        .transform(lambda x: (x / x.sum() * 100).round(1))
    )
    pivot_estado = estado_dist.pivot(index='cluster', columns='Estado', values='pct').fillna(0)
    print('Distribución Estado (%) por cluster:')
    print(pivot_estado)

    pivot_estado.plot(kind='bar', figsize=(8, 4), colormap='Set2', edgecolor='white')
    plt.title('Estado del contrato por cluster (%)')
    plt.xlabel('Cluster'); plt.ylabel('%')
    plt.legend(title='Estado', bbox_to_anchor=(1.05, 1))
    plt.xticks(rotation=0)
    plt.tight_layout(); plt.show()
else:
    print('WARN: columna Estado no encontrada en df')


### Proxy LTV por cluster
> LTV ≈ ValorMensual × Antigüedad\_dias — estima el valor acumulado del cliente;
> útil para priorizar retención y asignar presupuesto de CRM

In [ ]:
if 'Antiguedad_dias' in df.columns:
    if 'ValorMensual' not in df.columns:
        df['ValorMensual'] = df['ValorTotalPlan'] / df['Cuotas'].replace(0, 1)
    df['ltv_proxy'] = df['ValorMensual'] * df['Antiguedad_dias']

    ltv_stats = (
        df.groupby('cluster')['ltv_proxy']
        .describe(percentiles=[.25, .5, .75])
        [['mean', '25%', '50%', '75%']]
        .round(0)
    )
    ltv_fmt = ltv_stats.map(lambda x: f'${x:,.0f}')
    print('Proxy LTV por cluster (ValorMensual x Antigüedad_días):')
    print(ltv_fmt)
else:
    print('WARN: Antiguedad_dias no calculada, ejecuta la celda anterior primero')


### Top 5 Departamentos por cluster
> Distribución geográfica — insumo para regionalización de campañas

In [ ]:
geo_col = next(
    (c for c in df.columns if any(k in c.upper() for k in ['DEPART', 'REGION_V', 'MUNICIPIO', 'CIUDAD'])),
    None
)
if geo_col is None:
    print('WARN: no se encontró columna geográfica')
    print('Columnas candidatas:', [c for c in df.columns if any(k in c.upper() for k in ['REG','DEP','MUN','CIU'])])
else:
    print(f'Columna geográfica: {geo_col}')
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    pal = ['#682F2F', '#B9C0C9', '#9F8A78']
    for ax, cl, color in zip(axes, sorted(df['cluster'].unique()), pal):
        top5 = (
            df[df['cluster'] == cl][geo_col]
            .value_counts(normalize=True)
            .head(5)
            .mul(100)
            .round(1)
        )
        top5.plot(kind='barh', ax=ax, color=color, edgecolor='white')
        ax.set_title(f'Cluster {cl}', fontsize=12)
        ax.set_xlabel('%'); ax.invert_yaxis()
    plt.suptitle(f'Top 5 {geo_col} por cluster', fontsize=13)
    plt.tight_layout(); plt.show()


## 6. Construcción de tabla resumen (df_res)

In [ ]:
df_res["tamano_grupo"] = df_kmeans.groupby("cluster").size().values
df_res

In [ ]:
# Evitar SettingWithCopyWarning — df_kmeans es vista de df_modelo
df_kmeans = df_kmeans.copy()

In [ ]:
df_kmeans["categoria_entidad"] = pd.cut(
    df_kmeans["tamano_grupo"],
    bins=[0, 500, 2000, 5000, float('inf')],
    labels=["Pequeña", "Mediana", "Grande", "Muy grande"]  # Muy grande: >5000
)

In [ ]:
df_kmeans["valor_promedio_plan"] = pd.qcut(
    df_kmeans["ValorTotal_scaled"],
    q=3,
    labels=["Bajo", "Medio", "Alto"]
)

In [ ]:
def clasificar_generacion(anio):
    if 1928 <= anio <= 1945:
        return "Generación Silenciosa"
    elif 1946 <= anio <= 1964:
        return "Baby Boomers"
    elif 1965 <= anio <= 1980:
        return "Generación X"
    elif 1981 <= anio <= 1996:
        return "Millennials"
    elif 1997 <= anio <= 2012:
        return "Generación Z"
    elif 2013 <= anio <= 2025:
        return "Generación Alpha"
    else:
        return "Desconocido"

df_modelo["generacion"] = df_modelo["Año_nacimiento"].apply(clasificar_generacion)

In [ ]:
# Columnas para cuartiles en tabla resumen
# Incluye variables del modelo (NB02) + variables de perfil
# cuotas: Q1=Q3=12 en todos los clusters — no discrimina
# ValorTotal_scaled: valores escalados ilegibles para negocio; usar ValorTotalPlan
# Cantidad_mascotas: Q1=Q3=1 en todos los clusters — no discrimina
cols_info = [
    "Edad",
    "Estrato",
    "tamano_grupo",
    "ValorTotalPlan",
]

cols_name = [
    "edad",
    "estrato",
    "tamano_grupo",
    "valor_total_pesos",
]

In [ ]:
df_res = pd.DataFrame()

for i, col in enumerate(cols_info):
    df_res[f"{cols_name[i]}_q1"] = (
        df_kmeans.groupby("cluster")[col].quantile(0.25).values
    )
    df_res[f"{cols_name[i]}_q50"] = (
        df_kmeans.groupby("cluster")[col].quantile(0.50).values
    )
    df_res[f"{cols_name[i]}_q3"] = (
        df_kmeans.groupby("cluster")[col].quantile(0.75).values
    )

df_res

In [ ]:
# generacion_moda — usa .values con sort_index para alinear con df_res (sin columna cluster aun)
df_res["generacion_moda"] = (
    df_modelo.groupby("cluster")["generacion"]
    .agg(lambda x: x.mode()[0])
    .sort_index()
    .values
)

df_res["valor_promedio_plan"] = (
    df_kmeans.groupby("cluster")["valor_promedio_plan"]
    .agg(lambda x: x.mode()[0])
    .values
)

df_res["categoria_entidad"] = (
    df_kmeans.groupby("cluster")["categoria_entidad"]
    .agg(lambda x: x.mode()[0])
    .values
)

In [ ]:
binarias = [
    "TienePadres_V",
    "TieneEsposa_V",
    "TieneHijos_V"
]

for col in binarias:
    df_res[f"{col}_pct"] = (
        df_kmeans.groupby("cluster")[col].mean().values
    )

In [ ]:
df_res["% Con pareja"] = (df_res["TieneEsposa_V_pct"] * 100).round(1)
df_res["% Con hijos"] = (df_res["TieneHijos_V_pct"] * 100).round(1)
df_res["% Con padres"] = (df_res["TienePadres_V_pct"] * 100).round(1)

In [ ]:
df_res = df_res.drop(columns=[
    "TieneEsposa_V_pct",
    "TieneHijos_V_pct",
    "TienePadres_V_pct"
])

In [ ]:
# Variables geográficas — región, departamento y ciudad dominantes por cluster
for geo_col, prefix in [("REGION", "region"), ("DEPARTAMENTO", "departamento"), ("Ciudad", "ciudad")]:
    col_src = next((c for c in df_modelo.columns if c.upper() == geo_col.upper()), None)
    if col_src:
        df_res[f"{prefix}_top"] = (
            df_modelo.groupby("cluster")[col_src]
            .agg(lambda x: x.mode()[0] if len(x) > 0 else "N/A")
            .sort_index()
            .values
        )
        df_res[f"{prefix}_pct"] = (
            df_modelo.groupby("cluster")[col_src]
            .agg(lambda x: round(x.value_counts(normalize=True).iloc[0] * 100, 1) if len(x) > 0 else 0.0)
            .sort_index()
            .values
        )
print("Columnas geográficas agregadas:", [c for c in df_res.columns if any(c.startswith(p) for p in ["region_", "departamento_", "ciudad_"])])
df_res

In [ ]:
df_res.index.name = "cluster"
df_res = df_res.reset_index()

In [ ]:
# Tamaño por cluster
cluster_size = (
    df_kmeans["cluster"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

# Convertir a DataFrame
df_cluster_pct = cluster_size.reset_index()
df_cluster_pct.columns = ["cluster", "%_cluster"]
df_cluster_pct["%_cluster"] = df_cluster_pct["%_cluster"].round(2)

In [ ]:
df_res = df_res.merge(df_cluster_pct, on="cluster", how="left")

In [ ]:
# Ordenar clusters por porcentaje descendente
df_res = df_res.sort_values("%_cluster", ascending=False).reset_index(drop=True)

In [ ]:
# ── Naming comercial por cluster ─────────────────────────────────────
# Perfil: Generación + composición familiar + valor de contrato + producto
nombres_cluster = {
    0: 'Adultos Consolidados',        # Gen X mayor, sin familia, INTEGRAL, valor medio
    1: 'Familias de Alto Valor',         # Gen X, 84% con familia, UNIFAMILIAR+DIEZ, valor alto
    2: 'Jóvenes con Potencial',  # Millennials, sin familia, UNIPERSONAL, valor bajo
}
df_res['nombre_segmento'] = df_res['cluster'].map(nombres_cluster)
print('Naming comercial asignado:')
print(df_res[['cluster', 'nombre_segmento', '%_cluster']].to_string(index=False))


In [ ]:
# Exportar tabla resumen — requerida por app.py
df_res.to_excel('df_res.xlsx', index=False)
print(f'df_res.xlsx guardado | Shape: {df_res.shape}')
df_res

## 7. Enriquecimiento del dataset y exportación

In [ ]:
df_modelo = df_modelo.merge(
    df_res,
    on="cluster",
    how="left"
)

In [ ]:
columnas_no_sensibles = [
    # Identificación
    "Contrato", "NroEntidad", "Entidad",
    # Demográficas
    "Edad", "Sexo", "EstadoCivil", "Estrato",
    "TienePadres", "TieneEsposa", "TieneHijos",
    # Geográficas
    "DEPARTAMENTO", "REGION", "Region_V",
    # Variables binarias codificadas
    "TienePadres_V", "TieneEsposa_V", "TieneHijos_V", "EstadoCivil_V",
    # Actividad económica
    "SECTOR_EMPLEADOR", "ACTIVIDAD_ECONOMICA",
    # Producto y plan
    "Producto", "Canal", "ValorMensual", "Cuotas", "Act.valor",
    "ValorTotalPlan",
    # Variables del modelo K-Prototypes (NB02)
    "Cantidad_mascotas",
    "ValorTotal_scaled",
    # Derivadas
    "tamano_grupo", "Año_nacimiento",
    # Cuartiles de tabla resumen (Q1, mediana, Q3)
    "edad_q1", "edad_q50", "edad_q3",
    "estrato_q1", "estrato_q50", "estrato_q3",
    "tamano_grupo_q1", "tamano_grupo_q50", "tamano_grupo_q3",
    "valor_total_pesos_q1", "valor_total_pesos_q50", "valor_total_pesos_q3",
    # Categóricas enriquecidas
    "generacion", "generacion_moda", "valor_promedio_plan", "categoria_entidad",
    # Segmento
    "nombre_segmento",
    # Cluster
    "cluster"
]

df_modelo = df_modelo[columnas_no_sensibles].copy()

In [ ]:
df_modelo.to_excel("tabla_clusters.xlsx")

In [ ]:
df_modelo['generacion_moda'].value_counts()

## 8. Visualización comparativa — Radar por cluster
> Normalización min-max de 6 dimensiones clave para comparar perfiles de forma intuitiva

In [ ]:
import numpy as np

# Dimensiones numéricas del radar (deben existir en df_res)
dims_candidatos = [
    ('edad_q50',               'Edad\n(mediana)'),
    ('estrato_q50',            'Estrato\n(mediana)'),
    ('tamano_grupo_q50',       'Tamaño\ngrupo'),
    ('valor_total_pesos_q50',  'Valor total\n(mediana)'),
    ('% Con pareja',           '% Con\npareja'),
    ('% Con hijos',            '% Con\nhijos'),
]
dims   = [d for d, _ in dims_candidatos if d in df_res.columns]
labels = [l for d, l in dims_candidatos if d in df_res.columns]

if len(dims) < 3:
    print('WARN: menos de 3 dimensiones disponibles para el radar')
    print('Columnas numéricas en df_res:', df_res.select_dtypes('number').columns.tolist())
else:
    valores = df_res[dims].values.astype(float)
    mn, mx = valores.min(axis=0), valores.max(axis=0)
    mx = np.where(mx == mn, mn + 1, mx)
    valores_norm = (valores - mn) / (mx - mn)

    N = len(dims)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    # Paleta fija por número de cluster (no por posición de fila tras sort_values)
    _pal = {0: '#682F2F', 1: '#B9C0C9', 2: '#9F8A78'}
    colores = [_pal.get(c, '#888888') for c in df_res['cluster']]
    nombres  = (
        df_res['nombre_segmento'].tolist()
        if 'nombre_segmento' in df_res.columns
        else [f'Cluster {c}' for c in df_res['cluster']]
    )

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    for row, nombre, color in zip(valores_norm, nombres, colores):
        vals = row.tolist() + row[:1].tolist()
        ax.plot(angles, vals, 'o-', linewidth=2, color=color, label=nombre)
        ax.fill(angles, vals, alpha=0.15, color=color)

    ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8, color='grey')
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=11)
    ax.set_title('Perfil comparativo de clusters (normalizado)', size=14, pad=20)
    plt.tight_layout()
    plt.show()


## 9. Tabla de acciones comerciales por cluster
> Palancas de negocio diferenciadas por perfil para el equipo comercial

In [ ]:
acciones = {
    'Cluster':    [0, 1, 2],
    'Segmento':   ['Titulares Maduros', 'Familias Premium', 'Jóvenes en Crecimiento'],
    'Palanca':    [
        'Retención y fidelización',
        'Upselling y ampliación familiar',
        'Activación y primer upgrade',
    ],
    'Acción':     [
        'Campaña de renovación anticipada + beneficio exclusivo por antigüedad',
        'Oferta plan familiar extendido (agregar dependientes) + descuento por volumen',
        'Descuento primer año + pack bienvenida digital + recordatorio de chequeo',
    ],
    'KPI':        [
        'Tasa de renovación (meta >85%)',
        'Ticket promedio por contrato (meta +15%)',
        'Conversión plan base → plan plus (meta >20%)',
    ],
    'Canal':      [
        'Email + llamada personalizada',
        'Asesor comercial presencial',
        'App + WhatsApp + digital',
    ],
}
df_acciones = pd.DataFrame(acciones)
print('Tabla de acciones comerciales por cluster:')
display(df_acciones)